# Train a GuildLM Go specialist for **$0** on Kaggle

This notebook fine-tunes the **first GuildLM Code Guild Go specialist** — a QLoRA
adapter — using [`anvil`](https://github.com/guildlm/anvil), the GuildLM training
forge, on Kaggle's **free** T4 GPU.

**The free stack:** Kaggle (free GPU training) → HuggingFace Hub (free hosting) → Ollama (local serving).

**What it does, top to bottom:**
1. Install anvil's training stack + clone the `guild-code` recipe & dataset.
2. Load the committed Go SFT dataset with anvil's **real** `src.data` loaders.
3. Train a QLoRA LoRA adapter with anvil's **real** `src.train.train()` entrypoint.
4. Save the adapter to `/kaggle/working/go_reviewer_adapter`.
5. Quick sanity generation with the trained adapter.
6. *(Optional)* push the adapter to the HuggingFace Hub with `anvil-push`.

**Expected runtime:** ~5–15 min on a T4 with the tiny smoke-test sample dataset
and the 3B base. (A real teacher-generated dataset will take longer.)

**Before you Run All — two manual steps:**
- **Enable the GPU:** *Notebook → Settings → Accelerator → GPU T4 x2* (one T4 is enough).
- *(Optional, only for the push cell)* add your HuggingFace token under
  *Add-ons → Secrets* as `HF_TOKEN`.

> ⚠️ **Honesty note:** the bundled `code_guild_sample_v1` dataset is *offline-synthetic*
> (deterministic placeholders) — it exists to smoke-test the data→train pipeline
> end-to-end. For a shippable specialist, train on a real teacher-generated
> dataset (forge **online** mode). See `TRAINING.md`.


## 1. Configuration

All knobs live here. The **default base fits and finishes on a free 16 GB T4**.

> **Bigger GPU?** Bump `BASE_MODEL` to `Qwen/Qwen2.5-Coder-7B-Instruct` on a
> 24 GB+ card (RTX 3090/4090, A10, L4, A100). The 7B base is *too tight* to
> train comfortably on a free T4.


In [ ]:
# ---- Base model (T4-friendly default) ----------------------------------
# Default: a 3B coder base that QLoRA-trains comfortably on a free T4 (16 GB).
BASE_MODEL = "Qwen/Qwen2.5-Coder-3B-Instruct"
# 24 GB+ GPU?  ->  BASE_MODEL = "Qwen/Qwen2.5-Coder-7B-Instruct"

# ---- QLoRA / SFT hyper-parameters tuned to FINISH on a free T4 ----------
SEQ_LEN     = 1024   # short sequences = the biggest VRAM saving
BATCH_SIZE  = 1      # per-device micro-batch
GRAD_ACCUM  = 8      # effective batch = BATCH_SIZE * GRAD_ACCUM = 8
EPOCHS      = 2      # 1-2 epochs is plenty for the tiny sample dataset
LEARNING_RATE = 2.0e-4

# ---- Output -------------------------------------------------------------
OUTPUT_DIR = "/kaggle/working/go_reviewer_adapter"

# ---- Repos --------------------------------------------------------------
ANVIL_REPO      = "https://github.com/guildlm/anvil.git"
GUILD_CODE_REPO = "https://github.com/guildlm/guild-code.git"

print("Base model:", BASE_MODEL)
print("Adapter will be saved to:", OUTPUT_DIR)


## 2. Install anvil's training stack + clone the recipe & dataset

We **clone anvil** (so we get both the `src` package *and* the `configs/`
building blocks the recipe references) and editable-install its `[train]` extra
(torch, transformers, peft, trl, bitsandbytes, datasets, huggingface_hub).
We also clone **guild-code** for the `go_reviewer` recipe and the committed Go
dataset.


In [ ]:
import os, sys, subprocess, pathlib

WORK = pathlib.Path("/kaggle/working")
if not WORK.is_dir():           # not on Kaggle? fall back to the current dir
    WORK = pathlib.Path.cwd()

# If we are already *inside* the anvil repo (local dev), use it in place;
# otherwise clone it next to guild-code.
here = pathlib.Path.cwd()
if (here / "src" / "train.py").is_file() and (here / "configs").is_dir():
    ANVIL_DIR = here
    print("Running inside the anvil repo:", ANVIL_DIR)
else:
    ANVIL_DIR = WORK / "anvil"
    if not ANVIL_DIR.is_dir():
        subprocess.run(["git", "clone", "--depth", "1", ANVIL_REPO, str(ANVIL_DIR)], check=True)

GUILD_CODE_DIR = WORK / "guild-code"
if not GUILD_CODE_DIR.is_dir():
    subprocess.run(["git", "clone", "--depth", "1", GUILD_CODE_REPO, str(GUILD_CODE_DIR)], check=True)

print("anvil      ->", ANVIL_DIR)
print("guild-code ->", GUILD_CODE_DIR)


In [ ]:
# Install the training stack from the cloned anvil repo (editable).
# This is the path that ALSO gives us configs/ for recipe reference resolution.
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", f"{ANVIL_DIR}[train]"], check=True)

# --- Alternative (package only, no configs) -----------------------------
# If you only want the library and will supply configs yourself, you can instead:
#   !pip install -q "git+https://github.com/guildlm/anvil.git#egg=guildlm-anvil[train]"
# (We still need the cloned anvil/configs above for the recipe references.)

# Make `import src` resolve to the cloned anvil repo.
if str(ANVIL_DIR) not in sys.path:
    sys.path.insert(0, str(ANVIL_DIR))

print("anvil[train] installed.")


## 3. Resolve paths: recipe, configs root, and the committed dataset

The `go_reviewer` recipe references the `qwen2.5_7b` base block and the
`high_rank` LoRA block; those resolve against **anvil's** `configs/` root. The
dataset is the committed `code_guild_sample_v1` JSONL in guild-code.


In [ ]:
RECIPE_PATH   = str(GUILD_CODE_DIR / "go" / "anvil" / "go_reviewer.yaml")
CONFIGS_ROOT  = str(ANVIL_DIR / "configs")

DATA_DIR   = GUILD_CODE_DIR / "go" / "datasets" / "code_guild_sample_v1"
TRAIN_PATH = str(DATA_DIR / "code_guild_sample_v1.train.jsonl")
VAL_PATH   = str(DATA_DIR / "code_guild_sample_v1.validation.jsonl")

for p in (RECIPE_PATH, TRAIN_PATH, VAL_PATH):
    assert pathlib.Path(p).is_file(), f"missing: {p}"
print("recipe :", RECIPE_PATH)
print("configs:", CONFIGS_ROOT)
print("train  :", TRAIN_PATH)
print("val    :", VAL_PATH)


### Peek at the dataset with anvil's **real** `src.data` loader

`src.data.format_example` is the exact pure-python formatter anvil uses during
training. We render one row with the recipe's system prompt so you can see what
the model will actually be trained on.


In [ ]:
import json
from src.data import format_example

with open(TRAIN_PATH, "r", encoding="utf-8") as fh:
    first_row = json.loads(fh.readline())

print("Row schema keys:", sorted(first_row))
print("\n--- formatted training text (ChatML fallback preview) ---\n")
print(format_example(
    first_row,
    system_prompt="You are a meticulous senior Go engineer performing code review.",
)[:1200])


## 4. Train — anvil's real QLoRA SFT entrypoint

We load the recipe into anvil's real `AnvilConfig` via `load_recipe`, override a
handful of fields for the **free T4** (small base, short sequences, grad
accumulation, **fp16** because Turing T4s have no bf16, point at the committed
sample dataset), then hand it straight to **`src.train.train()`** — the same
function the `anvil-train` CLI calls. `train()` loads the base in 4-bit (QLoRA),
applies a single LoRA via TRL's `SFTTrainer`, and saves the adapter.


In [ ]:
from src.config import load_recipe
from src.train import train

# 1) Load the committed recipe (resolves base_model/lora refs vs anvil/configs).
recipe = load_recipe(RECIPE_PATH, configs_root=CONFIGS_ROOT)

# 2) Override for the free T4 + the committed sample dataset.
recipe.base_model.model_id           = BASE_MODEL
recipe.base_model.attn_implementation = None   # T4 has no FlashAttention-2
recipe.dataset.path        = TRAIN_PATH
recipe.dataset.eval_path   = VAL_PATH
recipe.dataset.val_split   = 0.0               # we have an explicit val file
recipe.dataset.max_seq_length = SEQ_LEN
recipe.output_dir          = OUTPUT_DIR

recipe.sft.batch_size                  = BATCH_SIZE
recipe.sft.gradient_accumulation_steps = GRAD_ACCUM
recipe.sft.epochs                      = EPOCHS
recipe.sft.learning_rate               = LEARNING_RATE
recipe.sft.bf16 = False                         # Turing T4: use fp16, not bf16
recipe.sft.fp16 = True

print("Effective seq length:", recipe.effective_max_seq_length)
print("Base model          :", recipe.base_model.model_id)
print("LoRA r / alpha      :", recipe.lora.r, "/", recipe.lora.alpha)
print("Output dir          :", recipe.output_dir)


In [ ]:
# This is the heavy step (loads the 4-bit base, runs SFT). ~5-15 min on a T4.
adapter_dir = train(recipe)
print("\nTrained LoRA adapter saved to:", adapter_dir)


In [ ]:
# Confirm the adapter files landed.
import os
print("Contents of", OUTPUT_DIR, ":")
for name in sorted(os.listdir(OUTPUT_DIR)):
    print("  ", name)


## 5. Sanity check — generate a Go review with the trained adapter

Load the 4-bit base + the LoRA adapter and generate a short review for a small
Go snippet. (With the offline-synthetic sample dataset the *content* will be
placeholder-flavoured — this cell verifies the **plumbing**: base + adapter load
and generate.)


In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

tok = AutoTokenizer.from_pretrained(BASE_MODEL)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

bnb = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.float16,
)
base = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL, quantization_config=bnb, device_map="auto",
)
model = PeftModel.from_pretrained(base, OUTPUT_DIR)
model.eval()

snippet = """package main

func Divide(a, b int) int {
    return a / b   // no zero check
}"""
messages = [
    {"role": "system", "content": "You are a meticulous senior Go engineer performing code review."},
    {"role": "user", "content": "Review this Go code:\n\n" + snippet},
]
prompt = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tok(prompt, return_tensors="pt").to(model.device)
with torch.no_grad():
    out = model.generate(**inputs, max_new_tokens=200, do_sample=False)
print(tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True))


## 6. *(Optional)* Push the adapter to the HuggingFace Hub — free hosting

This uses anvil's `anvil-push` helper (`src.hub.push_adapter`), which writes a
minimal model card and uploads the adapter folder. **Hosting on the Hub is free.**

**Setup:** add your HF token under *Add-ons → Secrets* as `HF_TOKEN`
(or paste it below). Create the token at https://huggingface.co/settings/tokens
with **write** access. Then set `HF_REPO_ID` and run the cell.


In [ ]:
# ---- EDIT THIS, then set PUSH = True to upload --------------------------
HF_REPO_ID = "your-username/go-reviewer-lora"   # <-- change me
PUSH = False                                    # <-- set True to actually push
PRIVATE = False
# ------------------------------------------------------------------------

if PUSH:
    import os
    # Prefer Kaggle Secrets, then env var.
    token = os.environ.get("HF_TOKEN")
    try:
        from kaggle_secrets import UserSecretsClient
        token = token or UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        pass

    from src.hub import push_adapter
    url = push_adapter(
        HF_REPO_ID, OUTPUT_DIR, token=token, private=PRIVATE, base_model=BASE_MODEL,
    )
    print("Pushed:", url)
else:
    print("PUSH is False — skipping. Set PUSH = True (and HF_REPO_ID) to upload.")
    print("CLI equivalent:")
    print(f"  anvil-push --repo-id {HF_REPO_ID} --adapter {OUTPUT_DIR} --base-model {BASE_MODEL}")


## Next steps

- **Merge + GGUF for local serving:** `anvil-merge` then `anvil-quantize --method gguf`,
  then run in **Ollama**. See [`TRAINING.md`](https://github.com/guildlm/anvil/blob/main/TRAINING.md).
- **Real quality:** swap the offline-synthetic sample for a real teacher-generated
  dataset (forge online mode) and re-run.
- **Bigger base:** on a 24 GB+ GPU set `BASE_MODEL = "Qwen/Qwen2.5-Coder-7B-Instruct"`.
